# M3 X1 on a Kaggle CPU: int8 export, fidelity, package sizes and quantised accuracy

We export every candidate model to ONNX in each precision, check each file against its fp32 export, and add up each
package against the 30 MB budget. This is the Kaggle half of X1: does the app fit the size budget, and what does
quantisation cost in accuracy? File sizes, int8 fidelity and int8 accuracy don't depend on the machine, so we measure
them here, on a Kaggle CPU kernel.

**Inputs:**
- the datasets `serpentguard-indian-snakes-clean` and `serpentguard-m3-subset`, found with
  `kagglehub.dataset_download`;
- the photo sample: the output of `SAMPLE_KERNEL`, i.e. `manifest.csv` with 200 "quant" training photos and 300 "val"
  validation photos, the photos at `data/<path>` beside it;
- each accuracy run: the output of its training kernel;
- the code in `../src` and Q1's retriever sizes in `../results/m3/q1/retrieval.csv`, which the runner kernel unpacks
  into `/kaggle/working/repo`.

Both kernel outputs are read through `kagglehub.notebook_output_download`, so declare each of those kernels as a
kernel source. Either can instead be a dataset, `"dataset:owner/slug"`, such as a private copy of another account's
run. A run's folder, named after its run id and holding `best.pt`, is found anywhere under it.

**Outputs** in `/kaggle/working/x1/`:
- `latency.csv`, `quantization.csv`, `package.csv`, `calibration_photos.csv` (subset paths only) and `versions.json`;
- `accuracy_<run_id>.json` per accuracy run.

ONNX files go to `/kaggle/tmp/x1_onnx`, and each model's are deleted once it is measured.

**Run:** on a Kaggle CPU kernel. `SPECS` limits the models, `SIZES` and `THREADS` set the input sizes and thread
counts, and `LATENCY_RUNS` the timed runs per file. `DETECTOR` is the fine-tuned detector: box-crop runs are scored on
its crops, and the matrix times it in place of an untrained YOLO26n.

**On the laptop**, set `SAMPLE_DIR`, `DATA_DIR`, `SUBSET_CSV` and `OUT_DIR`, and give `ACCURACY_RUNS` local run
folders; the pip cells then only check the pins. The ONNX files go to `<OUT_DIR>_onnx`.

## What the notebook measures

`run_matrix` exports the model matrix with untrained weights at 224 and 320 px (and
the detectors at 640), in each variant:
- int8 compute: static, or for the transformer encoders (MobileViT-S, DINOv2) static with attention kept fp32 and
  dynamic;
- int8_weights and fp16, which keep activations in float. Trained weights break int8 compute, so these are the
  deployment candidates.

Every variant is checked against its fp32 file on the sample's validation photos and timed; a variant that fails is
recorded with its reason and the run goes on. `package.csv` sums each precision's package against the 30 MB budget,
with an extra total and fits column for each Q1 retriever. `run_accuracy` then exports each run in
`ACCURACY_RUNS` from its `best.pt` in the same variants and scores each file with `quantized_accuracy` on the
validation photos, against the run's full validation counts.

**Latency on Kaggle is the secondary reference**, labelled "Kaggle CPU" in every latency row, in `package.csv` and in
`versions.json`. On the laptop, the O6 device, it is the primary latency, labelled "laptop CPU (`CPU_MODEL`)"; set
`CPU_MODEL` there, as `/proc/cpuinfo`, which gives it on Kaggle, doesn't exist on macOS.

In [1]:
SAMPLE_KERNEL = "puneetbajajiitm/sg-m3-local-sample"  # kernel "owner/slug" or "dataset:owner/slug": manifest, data/
ACCURACY_RUNS = [['../../kaggle/m3_runs/stageC/final_candidates/top1_s1/H1_top1_s1', 'H1_top1_s1']]
CPU_MODEL = 'Apple M4 Pro'
SPECS = ['snakeid_mobilenetv4_conv_small', 'snakeid_efficientnet_b0', 'snakeid_mobilevit_s', 'yolo26n-ft']
SIZES = [320]
THREADS = [1, 4]
LATENCY_RUNS = 100
DETECTOR = '../../kaggle/m3_runs/frontends/weights/frontends/D1/yolo26n-ft/detector.pt'
SAMPLE_DIR = '../../kaggle/m3_runs/local_sample/out2/local_sample'
DATA_DIR = '../../kaggle/clean_build/out'
SUBSET_CSV = '../../kaggle/m3_subset/subset.csv'
OUT_DIR = '../results/m3/x1_laptop'

In [2]:
import os
import subprocess
import sys
import warnings

os.environ["YOLO_VERBOSE"] = "False"  # Ultralytics' export log prints local file paths
warnings.filterwarnings("ignore", message="IProgress not found")  # tqdm wants ipywidgets

ALBUMENTATIONS_VERSION = "2.0.8"  # requirements-m3.txt; calibration photos go through its eval normalisation
TIMM_VERSION = "1.0.31"            # requirements-m3.txt; the encoders
ONNX_VERSION = "1.23.2"            # requirements-m3.txt
ONNXRUNTIME_VERSION = "1.31.1"     # requirements-m3.txt; quantisation, the fp32-op list and latency depend on it
ONNXSCRIPT_VERSION = "0.7.2"       # requirements-m3.txt; F5's export through torch.export
ULTRALYTICS_VERSION = "8.4.202"    # requirements-m3.txt; YOLO26n-seg, YOLO26n and yolov8n-oiv7 export through it
SMP_VERSION = "0.6.0"              # requirements-m3.txt; segmentation-models-pytorch, the U-Net
if os.path.exists("/kaggle/working"):  # the laptop's venv already holds these pins; the asserts below check them
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"albumentations=={ALBUMENTATIONS_VERSION}",
                    f"timm=={TIMM_VERSION}", f"onnx=={ONNX_VERSION}", f"onnxruntime=={ONNXRUNTIME_VERSION}",
                    f"onnxscript=={ONNXSCRIPT_VERSION}", f"ultralytics=={ULTRALYTICS_VERSION}",
                    f"segmentation-models-pytorch=={SMP_VERSION}"], check=True)
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"  # otherwise albumentations asks PyPI for its latest version on import

import albumentations
import onnx
import onnxruntime
import onnxscript
import segmentation_models_pytorch
import timm
import torch
import ultralytics

assert albumentations.__version__ == ALBUMENTATIONS_VERSION, f"albumentations {albumentations.__version__} is loaded"
assert timm.__version__ == TIMM_VERSION, f"timm {timm.__version__} is loaded"
assert onnx.__version__ == ONNX_VERSION, f"onnx {onnx.__version__} is loaded"
assert onnxruntime.__version__ == ONNXRUNTIME_VERSION, f"onnxruntime {onnxruntime.__version__} is loaded"
assert onnxscript.__version__ == ONNXSCRIPT_VERSION, f"onnxscript {onnxscript.__version__} is loaded"
assert ultralytics.__version__ == ULTRALYTICS_VERSION, f"ultralytics {ultralytics.__version__} is loaded"
assert segmentation_models_pytorch.__version__ == SMP_VERSION, (f"smp {segmentation_models_pytorch.__version__} "
                                                                f"is loaded")
print(f"albumentations {albumentations.__version__}, timm {timm.__version__}, onnx {onnx.__version__}, "
      f"onnxruntime {onnxruntime.__version__}, onnxscript {onnxscript.__version__}, "
      f"ultralytics {ultralytics.__version__}, smp {segmentation_models_pytorch.__version__}, "
      f"torch {torch.__version__}")

albumentations 2.0.8, timm 1.0.31, onnx 1.23.2, onnxruntime 1.31.1, onnxscript 0.7.2, ultralytics 8.4.202, smp 0.6.0, torch 2.15.0


In [3]:
SCIKIT_LEARN_VERSION = "1.9.1"  # requirements-m3.txt; quantized_accuracy's metrics use it
if os.path.exists("/kaggle/working"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"scikit-learn=={SCIKIT_LEARN_VERSION}"], check=True)

import sklearn

assert sklearn.__version__ == SCIKIT_LEARN_VERSION, (f"scikit-learn {sklearn.__version__} is loaded; restart the "
                                                     f"kernel after the install")
print(f"scikit-learn {sklearn.__version__}")

scikit-learn 1.9.1


In [4]:
import json
import platform
import time
from pathlib import Path

CODE = Path("/kaggle/input/serpentguard-code/src")
CODE = CODE if CODE.exists() else Path("../src").resolve()
sys.path.insert(0, str(CODE))

import pandas as pd

from serpentguard.run_files import CHECKPOINT_FILE

## The X1 code

The next three cells hold what used to be `serpentguard.export` and `serpentguard.export_matrix`: converting a model
and checking each copy, the model matrix with its package tables, and scoring a trained run after conversion.
Trained weights break int8 activations:
a SiLU output peaking near 100 where most values are under 1, and depthwise weights
spread up to 46x between channels,
so int8_weights and fp16 keep activations in float.

In [5]:
import glob
import json
import os
import platform
import time
import warnings
from collections import Counter, defaultdict
from contextlib import contextmanager
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Callable

import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
import torch
from onnx import TensorProto
from onnxruntime.quantization import (CalibrationDataReader, CalibrationMethod, QuantFormat, QuantType,
                                      quantize_dynamic, quantize_static)
from onnxruntime.quantization.shape_inference import quant_pre_process
from onnxruntime.transformers.float16 import convert_float_to_float16
from torch import nn

from serpentguard.data import AUGMENT, FRONT_END_MODES, apply_mode, is_missing_box, load_rgb
from serpentguard.export import open_session, runtime_graph
from serpentguard.frontend import FINE_TUNED, OFF_THE_SHELF, predict_boxes
from serpentguard.frontend_scores import YOLO_CONF
from serpentguard.models import DINOV2, DINOV2_PATCH, EMBEDDING_DIM, MASK, MOBILE_ENCODERS, N_FAMILY, N_GENUS, SnakeID
from serpentguard.run_data import BOX_CROP, SAM, eval_loader, input_mode, read_tables, run_frames
from serpentguard.run_files import PROTOTYPES_FILE
from serpentguard.run_outputs import (class_tables, final_metrics, library_versions, load_checkpoint, subset_hash,
                                      write_json)
from serpentguard.segmenters import TORCH_KINDS, YOLO_SEG, build_segmenter
from serpentguard.train import VAL_OUTPUTS

OPSET, DYNAMO_OPSET = 17, 18
TORCHSCRIPT, DYNAMO = "torchscript", "dynamo"
SNAKEID_OUTPUTS = ("species_logits", "embedding", "who_logits", "genus_logits", "family_logits")
SPECIES, EMBEDDING = SNAKEID_OUTPUTS[:2]
IMAGENET, UNIT = "imagenet", "unit"   # SnakeID and the torch segmenters read ImageNet-normalised input, YOLO 0-1 RGB
STATIC, STATIC_ATTENTION_FP32, DYNAMIC = "static_qdq", "static_qdq_attention_fp32", "dynamic"
INT8_WEIGHTS, FP16 = "int8_weights", "fp16"
INT8_SUFFIXES = {STATIC: ".int8.onnx", STATIC_ATTENTION_FP32: ".attention_fp32.int8.onnx",
                 DYNAMIC: ".dynamic.int8.onnx", INT8_WEIGHTS: ".int8_weights.onnx"}
WEIGHT_OPS = ("Conv", "MatMul", "Gemm")  # the operators whose weight (input 1) int8_weights stores in int8
INT8_LIMIT = 127
ATTENTION_KEY = "attention_nodes_kept_fp32"
QUANTISATION_OPS = ("QuantizeLinear", "DequantizeLinear", "DynamicQuantizeLinear")
INTEGER_OP_MARKERS = ("Integer", "QLinear", "QuantizeMatMul", "QGemm", "QAttention")  # integer kernels' names
FLOAT_TYPES = (TensorProto.FLOAT, TensorProto.FLOAT16, TensorProto.BFLOAT16, TensorProto.DOUBLE,
               TensorProto.UNDEFINED)   # UNDEFINED: a type shape inference couldn't work out, counted as float
CALIBRATION_FLUSH = 16      # photos whose activations onnxruntime holds before folding them into the ranges
STATIC_OPTIONS = {"quant_format": QuantFormat.QDQ, "per_channel": True, "activation_type": QuantType.QInt8,
                  "weight_type": QuantType.QInt8, "calibrate_method": CalibrationMethod.MinMax}
MEGABYTE = 10 ** 6
WARMUP_RUNS = 10  # untimed runs before timing; LATENCY_RUNS comes from the parameters cell


class DeployedHeads(nn.Module):
    """SnakeID's outputs as a tuple, in `names` order."""

    def __init__(self, model, names):
        super().__init__()
        self.model, self.names = model, tuple(names)

    def forward(self, images):
        outputs = self.model(images)
        return tuple(outputs[name] for name in self.names)


def to_onnx(model, path, input_size):
    """Exports SnakeID or a torch segmenter with a batch axis; F5 needs torch.export, at batch 1."""
    path = Path(path)
    model.eval()
    with torch.no_grad():
        sample = model(torch.zeros(1, 3, input_size, input_size))
    if isinstance(sample, dict):
        names = [name for name in SNAKEID_OUTPUTS if sample[name] is not None]
        net = DeployedHeads(model, names).eval()
    else:
        names, net = ["mask_logits"], model
    try:
        _export(net, path, names, input_size, dynamo=False)
        exporter = TORCHSCRIPT
    except torch.onnx.OnnxExporterError as error:
        _export(net, path, names, input_size, dynamo=True)
        exporter = f"{DYNAMO} (TorchScript: {_first_line(error)})"
    _set_props(path, {"exporter": exporter})
    return path


def _export(net, path, names, input_size, dynamo):
    options = ({"dynamo": True, "opset_version": DYNAMO_OPSET, "external_data": False, "verbose": False} if dynamo
               else {"dynamo": False, "opset_version": OPSET,
                     "dynamic_axes": {name: {0: "batch"} for name in ("images", *names)}})
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", DeprecationWarning)
        warnings.simplefilter("ignore", torch.jit.TracerWarning)
        torch.onnx.export(net, (torch.zeros(1, 3, input_size, input_size),), str(path), input_names=["images"],
                          output_names=names, **options)


def _first_line(error):
    return f"{type(error).__name__}: {(str(error).splitlines() or [''])[0]}"


def _props(path):
    return {entry.key: entry.value for entry in onnx.load(str(path), load_external_data=False).metadata_props}


def _set_props(path, values):
    model = onnx.load(str(path))
    onnx.helper.set_model_props(model, {**{entry.key: entry.value for entry in model.metadata_props}, **values})
    onnx.save(model, str(path))


def _sibling(path, suffix):
    return path.with_name(path.name.removesuffix(".onnx") + suffix)


def exporter_of(path):
    return _props(path).get("exporter", "")


def opset_of(path):
    imports = onnx.load(str(path), load_external_data=False).opset_import
    return next(entry.version for entry in imports if entry.domain in ("", "ai.onnx"))


def calibration_inputs(files, input_size, normalisation=IMAGENET, boxes=None):
    """(N, 3, size, size) float32 inputs as inference reads them: box crop if given, letterbox, then normalisation."""
    files = list(files)
    boxes = [None] * len(files) if boxes is None else list(boxes)
    canvases = [apply_mode(load_rgb(file), None, FRONT_END_MODES["F1"], box, input_size)[0]
                for file, box in zip(files, boxes)]
    if normalisation == IMAGENET:
        return np.stack([AUGMENT["eval"](image=canvas)["image"].numpy() for canvas in canvases]).astype(np.float32)
    return np.stack([canvas.transpose(2, 0, 1) / 255.0 for canvas in canvases]).astype(np.float32)


class ArrayReader(CalibrationDataReader):
    """Calibration photos for onnxruntime, one per run: the batch size the phone runs."""

    def __init__(self, name, inputs):
        self.feeds = ({name: inputs[index:index + 1]} for index in range(len(inputs)))

    def get_next(self):
        return next(self.feeds, None)


def quantize_int8(path, calibration, out_path=None, variant=STATIC):
    """int8 copy of an fp32 ONNX file; a static variant that raises falls back to dynamic."""
    path, calibration = Path(path), np.asarray(calibration, dtype=np.float32)
    if not len(calibration):
        raise ValueError("int8 needs calibration photos; got none")
    out = Path(out_path) if out_path else _sibling(path, INT8_SUFFIXES[variant])
    record = {"variant": variant, "quantization": variant, "fallback_reason": "", ATTENTION_KEY: "0"}
    if variant == DYNAMIC:
        _quantize_dynamic(path, out, calibration)
    elif variant == INT8_WEIGHTS:
        with _prepared(path, out) as prepared:
            model = onnx.load(str(prepared))
            _int8_weights(model)
            onnx.save(model, str(out))
        run_photos(out, calibration[:1])
    else:
        try:
            record[ATTENTION_KEY] = str(_quantize_static(path, out, calibration, variant == STATIC_ATTENTION_FP32))
        except Exception as error:  # the fallback is defined by failure: whatever static QDQ can't quantise or run
            if resource_error(error):
                raise
            _quantize_dynamic(path, out, calibration)
            record.update({"quantization": DYNAMIC, "fallback_reason": _first_line(error)})
    return _record(path, out, record)


def resource_error(error):
    """A full disk or exhausted memory: the machine's error, not the model's."""
    return isinstance(error, (OSError, MemoryError)) or "No space left on device" in str(error)


def _record(source, out, record, float_types=FLOAT_TYPES):
    """Writes into out's metadata the source's, the record, and the nodes the CPU provider still runs in float."""
    left = fp32_nodes(out, float_types)
    _set_props(out, {**_props(source), **record,
                     "fp32_ops": json.dumps(dict(sorted(Counter(op_type for _, op_type in left).items()))),
                     "fp32_nodes": json.dumps([name for name, _ in left])})
    return out


@contextmanager
def _prepared(path, out):
    """quant_pre_process's copy of path beside out, deleted afterwards with onnxruntime's inferred copy."""
    prepared = _sibling(out, ".pre.onnx")
    try:
        try:
            quant_pre_process(str(path), str(prepared))
        except Exception as error:  # symbolic shape inference raises bare TypeErrors on graphs it can't follow (F5)
            if resource_error(error):
                raise
            quant_pre_process(str(path), str(prepared), skip_symbolic_shape=True)
        yield prepared
    finally:
        prepared.unlink(missing_ok=True)
        _sibling(prepared, "-inferred.onnx").unlink(missing_ok=True)


def _quantize_static(path, out, calibration, attention_fp32):
    """Static QDQ, with the attention nodes left fp32 when attention_fp32; returns how many were."""
    with _prepared(path, out) as prepared:
        excluded = []
        if attention_fp32:
            model = onnx.load(str(prepared))
            unnamed = [(index, node) for index, node in enumerate(model.graph.node) if not node.name]
            for index, node in unnamed:  # the quantiser skips nodes by name
                node.name = f"{node.op_type}_unnamed_{index}"
            if unnamed:
                onnx.save(model, str(prepared))
            excluded = attention_nodes(model)
        name = open_session(prepared).get_inputs()[0].name
        options = {**STATIC_OPTIONS, "extra_options": {"CalibMaxIntermediateOutputs": calibration_flush(
            len(calibration))}}
        quantize_static(str(prepared), str(out), ArrayReader(name, calibration), nodes_to_exclude=excluded, **options)
    if excluded:
        _bypass_dequantize(out, excluded)
    run_photos(out, calibration[:1])
    return len(excluded)


def _bypass_dequantize(path, nodes):
    """Feeds the named nodes the float tensors behind their Quantize -> Dequantize pairs, which onnxruntime would
    otherwise fuse into integer kernels at load time."""
    model = onnx.load(str(path))
    producer = {output: node for node in model.graph.node for output in node.output}
    names, rewired = set(nodes), 0
    for node in model.graph.node:
        if node.name not in names:
            continue
        for index, tensor in enumerate(node.input):
            dequantize = producer.get(tensor)
            quantize = producer.get(dequantize.input[0]) if dequantize and dequantize.op_type == "DequantizeLinear" \
                else None
            if quantize is not None and quantize.op_type == "QuantizeLinear":  # a weight's reads a constant
                node.input[index] = quantize.input[0]
                rewired += 1
    if rewired:
        onnx.save(model, str(path))


def calibration_flush(photos, largest=CALIBRATION_FLUSH):
    """The largest flush size up to `largest` that doesn't divide the photo count: onnxruntime's MinMax calibrator
    raises "No data is collected" when the last flush leaves its buffer empty."""
    return next((size for size in range(largest, 1, -1) if photos % size), largest)


def _quantize_dynamic(path, out, calibration):
    """Dynamic int8; value_info goes, as the quantiser transposes a Gemm weight that a declared shape contradicts."""
    with _prepared(path, out) as prepared:
        model = onnx.load(str(prepared))
        del model.graph.value_info[:]
        onnx.save(model, str(prepared))
        quantize_dynamic(str(prepared), str(out), weight_type=QuantType.QInt8)
    run_photos(out, calibration[:1])


def _int8_weights(model):
    """Stores each float weight of a WEIGHT_OPS node as symmetric int8, one scale per output channel."""
    graph = model.graph
    initializers = {tensor.name: tensor for tensor in graph.initializer}
    stored, dequantize = {}, []
    for node in graph.node:
        name = node.input[1] if node.op_type in WEIGHT_OPS and len(node.input) > 1 else None
        tensor = initializers.get(name)
        if tensor is None or name in stored or tensor.data_type != TensorProto.FLOAT or len(tensor.dims) < 2:
            continue
        weight = onnx.numpy_helper.to_array(tensor)
        transposed = node.op_type == "Gemm" and any(attr.name == "transB" and attr.i for attr in node.attribute)
        axis = 0 if node.op_type == "Conv" or transposed else weight.ndim - 1  # the output channels
        scale = np.abs(weight).max(axis=tuple(i for i in range(weight.ndim) if i != axis)) / INT8_LIMIT
        scale = np.where(scale > 0, scale, 1.0).astype(np.float32)
        shape = [1] * weight.ndim
        shape[axis] = -1
        values = np.clip(np.round(weight / scale.reshape(shape)), -INT8_LIMIT, INT8_LIMIT).astype(np.int8)
        names = [name + suffix for suffix in ("_int8", "_scale", "_zero_point")]
        stored[name] = [onnx.numpy_helper.from_array(array, label) for array, label in
                        zip((values, scale, np.zeros_like(scale, dtype=np.int8)), names)]
        dequantize.append(onnx.helper.make_node("DequantizeLinear", names, [name], name=f"{name}_dequantize",
                                                axis=axis))
    kept = [tensor for tensor in graph.initializer if tensor.name not in stored]
    del graph.initializer[:]
    graph.initializer.extend(kept + [tensor for tensors in stored.values() for tensor in tensors])
    nodes = list(graph.node)
    del graph.node[:]
    graph.node.extend(dequantize + nodes)


def to_fp16(path, out_path=None):
    """fp16 copy of an fp32 ONNX file, input and outputs kept float32."""
    path = Path(path)
    out = Path(out_path) if out_path else _sibling(path, ".fp16.onnx")
    onnx.save(convert_float_to_float16(onnx.load(str(path)), keep_io_types=True), str(out))
    return _record(path, out, {"variant": FP16, "quantization": FP16, "fallback_reason": "", ATTENTION_KEY: "0"},
                   (TensorProto.FLOAT,))


def attention_nodes(model):
    """Names of every Softmax, LayerNormalization and MatMul of two activations; a MatMul with a weight isn't one."""
    graph = model.graph
    constants = {tensor.name for tensor in graph.initializer} | {
        output for node in graph.node if node.op_type == "Constant" for output in node.output}
    return [node.name for node in graph.node if node.op_type in ("Softmax", "LayerNormalization")
            or (node.op_type == "MatMul" and not constants.intersection(node.input))]


def fp32_nodes(path, float_types=FLOAT_TYPES):
    """(name, op_type) of each node the CPU provider runs in float after its load-time fusions."""
    model = runtime_graph(path)
    declared = _declared_types(model.graph)
    try:
        inferred = _declared_types(onnx.shape_inference.infer_shapes(model).graph)
    except onnx.shape_inference.InferenceError:
        inferred = {}
    types = {**inferred, **{name: kind for name, kind in declared.items() if kind != TensorProto.UNDEFINED}}
    return [(node.name, node.op_type) for node in model.graph.node
            if node.input and not (node.op_type in QUANTISATION_OPS
                                   or any(marker in node.op_type for marker in INTEGER_OP_MARKERS))
            and any(types.get(output, TensorProto.UNDEFINED) in float_types for output in node.output)]


def _declared_types(graph):
    return {info.name: info.type.tensor_type.elem_type for info in (*graph.value_info, *graph.output, *graph.input)}


def quantization_of(path):
    """The file's variant, quantization, fallback_reason, fp32_ops and attention_nodes_kept_fp32."""
    props = _props(path)
    return {"variant": props.get("variant", ""), "quantization": props.get("quantization", ""),
            "fallback_reason": props.get("fallback_reason", ""), "fp32_ops": json.loads(props.get("fp32_ops", "{}")),
            ATTENTION_KEY: int(props.get(ATTENTION_KEY, 0))}


def fp32_node_names(path):
    return json.loads(_props(path).get("fp32_nodes", "[]"))


def run_photos(path, inputs, threads=None):
    """{output name: (N, ...) array} of an ONNX model, one photo per run (some files take batch 1 only)."""
    return _run_session(open_session(path, threads), inputs)


def _run_session(session, inputs):
    name, outputs = session.get_inputs()[0].name, [output.name for output in session.get_outputs()]
    runs = [session.run(None, {name: inputs[index:index + 1]}) for index in range(len(inputs))]
    return {output: np.concatenate([run[position] for run in runs]) for position, output in enumerate(outputs)}


def compare_outputs(reference, quantized):
    """Per-output correlation and largest difference, the lowest correlation, and top-1 agreement on species."""
    outputs = {name: {"corr": _correlation(reference[name], quantized[name]),
                      "max_abs_diff": float(np.abs(reference[name] - quantized[name]).max())} for name in reference}
    agreement = (float(np.mean(reference[SPECIES].argmax(axis=1) == quantized[SPECIES].argmax(axis=1)))
                 if SPECIES in reference else None)
    return {"min_corr": min(entry["corr"] for entry in outputs.values()), "top1_agreement": agreement,
            "outputs": outputs}


def _correlation(first, second):
    """Pearson correlation; a constant side scores 1 when both sides are equal and 0 otherwise."""
    first, second = np.ravel(first).astype(np.float64), np.ravel(second).astype(np.float64)
    if first.std() == 0 or second.std() == 0:
        return float(np.array_equal(first, second))
    return float(np.corrcoef(first, second)[0, 1])


def fidelity(reference, quantized, inputs):
    return compare_outputs(run_photos(reference, inputs), run_photos(quantized, inputs))


def file_mb(path):
    return Path(path).stat().st_size / MEGABYTE


def cpu_latency(path, input_size, runs=LATENCY_RUNS, threads=1, warmup=WARMUP_RUNS):
    """Batch-1 CPU latency in a fresh session after `warmup` untimed runs: median_ms, p95_ms, fps and mb."""
    session = open_session(path, threads)
    image = np.random.default_rng(0).random((1, 3, input_size, input_size), dtype=np.float32)
    feed = {session.get_inputs()[0].name: image}
    for _ in range(warmup):
        session.run(None, feed)
    times = np.empty(runs)
    for index in range(runs):
        started = time.perf_counter()
        session.run(None, feed)
        times[index] = (time.perf_counter() - started) * 1000.0
    median = float(np.median(times))
    return {"median_ms": median, "p95_ms": float(np.percentile(times, 95)), "fps": 1000.0 / median, "mb": file_mb(path)}

In [6]:
os.environ.setdefault("YOLO_OFFLINE", "1")  # Ultralytics builds and exports without the network
# SIZES and THREADS, the defaults below, come from the parameters cell

BUDGET_MB = 30.0            # the whole on-device package after int8 (M1 O7)
SNAKEID, SNAKEID_F5, SEGMENTER, DETECTOR_ROLE = "snakeid", "snakeid_f5", "segmenter", "detector"
NATIVE_DETECTOR_SIZE = 640
MS_PREFIX = "ms_"
PACKAGE_NOTE = "fits leaves out the Q1 retrieval index and its embedder"
N_CLASSES = 184                     # classes.csv
CALIBRATION_PHOTOS = 200
TRANSFORMER_ENCODERS = ("mobilevit_s", DINOV2)                # attention gets its own int8 variants
TWO_VARIANTS, ONE_VARIANT = (STATIC_ATTENTION_FP32, DYNAMIC), (STATIC,)  # int8 compute: activations in int8
DEPLOYMENT_VARIANTS = (INT8_WEIGHTS, FP16)  # activations stay float, which trained weights survive
FAILED = "failed"
INT8_PACKAGE = "int8"                       # package block of each model's chosen int8 compute variant
PACKAGE_PRECISIONS = (INT8_PACKAGE, INT8_WEIGHTS, FP16)
FIDELITY_MIN_CORR = 0.95            # untrained-weight min correlation an int8 variant needs to be packaged
YOLO_CONFIGS = {YOLO_SEG: ("yolo26n-seg.yaml", 1),          # S1's YOLO26n-seg, fine-tuned to one class
                FINE_TUNED: ("yolo26n.yaml", 1),             # D1's YOLO26n, fine-tuned to one class
                OFF_THE_SHELF: ("yolov8n.yaml", 601)}        # B5's YOLOv8n trained on Open Images V7's 601 classes
DETECTORS = (FINE_TUNED, OFF_THE_SHELF)
NEEDS_NMS = (OFF_THE_SHELF,)        # YOLOv8 needs NMS after the network; YOLO26's head is NMS-free
FULL_INPUT = FRONT_END_MODES["F0"]
DETECTOR_SIZE = 320                 # D1's YOLO26n was fine-tuned and is deployed at this input size
DETECTOR_CHUNK = 64
FP32, INT8, HALF = "fp32", "int8", "fp16"  # latency.csv's precision column
MANIFEST = "manifest.csv"
LATENCY_FILE, QUANTIZATION_FILE, PACKAGE_FILE = "latency.csv", "quantization.csv", "package.csv"
CALIBRATION_FILE, VERSIONS_FILE, ACCURACY_FILE = "calibration_photos.csv", "versions.json", "accuracy_{}.json"
WITH_RETRIEVER_MB, WITH_RETRIEVER_FITS = "total_mb_with_{}", "fits_with_{}"
EXPORTER = ("torch.onnx.export: TorchScript (dynamo=False), torch.export (dynamo=True) where TorchScript fails, per "
            "quantization.csv; Ultralytics' own exporter for YOLO, simplify=False")
DETECTION_BAR = 0.95                # detection_agreement a front end's precision needs to go in the deployment package
DETECTION_IOU = 0.5                 # top boxes overlapping this much are the same detection
DETECTION_COLUMNS = ("detection_agreement", "fp32_detection_rate", "both_found_agreement", "both_empty_share")
DEPLOYMENT = "deployment"           # package block: SnakeID at --snakeid-precision, each detector at its best passing
FPS_TARGET = 10                     # frames per second the live screen needs (O6)


@dataclass(frozen=True)
class ModelSpec:
    name: str
    role: str
    encoder: str
    size: int
    normalisation: str
    build: Callable[[Path], Path]    # writes the fp32 ONNX file at the path and returns it
    nominal_size: int
    weights: str = "untrained"       # or "trained: <file>" for fine-tuned YOLO weights


def model_specs(sizes=SIZES, n_classes=N_CLASSES, embedding_dim=EMBEDDING_DIM, trained=None):
    """The matrix, size by size: SnakeID per encoder (plain, and F5 for the mobile encoders), DINOv2, the four S1
    segmenters and the two detectors; then the detectors at 640. `trained` maps a YOLO model to fine-tuned weights."""
    trained = dict(trained or {})
    if set(trained) - set(YOLO_CONFIGS):
        raise ValueError(f"--trained takes the YOLO models {sorted(YOLO_CONFIGS)}; got {sorted(trained)}")
    specs = []
    for size in sizes:
        dino_size = -(-size // DINOV2_PATCH) * DINOV2_PATCH  # the patch multiple at or above size: 320 becomes 322
        for encoder in MOBILE_ENCODERS:
            specs.append(ModelSpec(f"snakeid_{encoder}", SNAKEID, encoder, size, IMAGENET,
                                   _snakeid(encoder, size, n_classes, embedding_dim, False), size))
            specs.append(ModelSpec(f"snakeid_f5_{encoder}", SNAKEID_F5, encoder, size, IMAGENET,
                                   _snakeid(encoder, size, n_classes, embedding_dim, True), size))
        specs.append(ModelSpec(f"snakeid_{DINOV2}", SNAKEID, DINOV2, dino_size, IMAGENET,
                               _snakeid(DINOV2, dino_size, n_classes, embedding_dim, False), size))
        specs += [ModelSpec(kind, SEGMENTER, "", size, IMAGENET, _segmenter(kind, size), size) for kind in TORCH_KINDS]
        specs.append(ModelSpec(YOLO_SEG, SEGMENTER, "", size, UNIT, _yolo(YOLO_SEG, size), size))
        specs += [ModelSpec(name, DETECTOR_ROLE, "", size, UNIT, _yolo(name, size), size) for name in DETECTORS]
    specs += [ModelSpec(name, DETECTOR_ROLE, "", NATIVE_DETECTOR_SIZE, UNIT, _yolo(name, NATIVE_DETECTOR_SIZE),
                        NATIVE_DETECTOR_SIZE) for name in DETECTORS]
    return [spec if spec.name not in trained else replace(spec, build=_yolo(spec.name, spec.size, trained[spec.name]),
                                                           weights=f"trained: {Path(trained[spec.name]).name}")
            for spec in specs]


def _snakeid(encoder, size, n_classes, embedding_dim, f5):
    def build(path):
        torch.manual_seed(0)
        options = {"mask_head": True, "pooling": MASK} if f5 else {}
        model = SnakeID(encoder, n_classes, N_GENUS, N_FAMILY, embedding_dim=embedding_dim, pretrained=False,
                        input_size=size, **options)
        return to_onnx(model, path, size)
    return build


def _segmenter(kind, size):
    def build(path):
        torch.manual_seed(0)
        return to_onnx(build_segmenter(kind, pretrained=False, input_size=size), path, size)
    return build


def _yolo(name, size, weights=None):
    def build(path):
        torch.manual_seed(0)
        return ultralytics_onnx(weights or YOLO_CONFIGS[name][0], YOLO_CONFIGS[name][1], path, size)
    return build


def ultralytics_onnx(config, n_classes, path, input_size):
    """An Ultralytics yaml or .pt with n_classes, through Ultralytics' own exporter (batch 1, fixed size, no slim)."""
    from ultralytics import YOLO

    yolo = YOLO(config)
    if yolo.model.yaml["nc"] != n_classes:
        model = type(yolo.model)({**yolo.model.yaml, "nc": n_classes}, verbose=False)
        model.args, model.task = yolo.model.args, yolo.model.task
        yolo.model = model
    path = Path(path)
    yolo.model.pt_path = str(path.with_suffix(".pt"))  # the exporter names its file after this
    exported = Path(yolo.export(format="onnx", imgsz=input_size, opset=OPSET, simplify=False, dynamic=False,
                                half=False, verbose=False))
    if exported.resolve() != path.resolve():
        exported.replace(path)
    return path


def read_sample(sample, calibration_photos=CALIBRATION_PHOTOS):
    """(calibration, held_out): the manifest's "quant" training photos and its "val" validation photos."""
    sample = Path(sample)
    manifest = pd.read_csv(sample / MANIFEST, dtype=str)
    manifest = manifest.assign(file=[str(sample / "data" / path) for path in manifest["path"]])
    calibration, held_out = (manifest[manifest["use"] == use][["file", "path", "species"]].reset_index(drop=True)
                             for use in ("quant", "val"))
    if len(calibration) != calibration_photos:
        raise ValueError(f"the sample has {len(calibration)} quant photos; static int8 calibrates on "
                         f"{calibration_photos}")
    for frame, prefix, use in ((calibration, "train/", "quant"), (held_out, "val/", "val")):
        outside = ~frame["path"].str.startswith(prefix)
        if frame.empty or outside.any():
            raise ValueError(f"every {use} photo must be under {prefix}; "
                             f"{'none are given' if frame.empty else frame.loc[outside, 'path'].iloc[0]}")
    return calibration, held_out


@contextmanager
def model_files(onnx_root, stem):
    """On the way out deletes every <stem>.* or <stem>-* file: a model's files go once it is measured."""
    try:
        yield
    finally:
        for leftover in onnx_root.glob(f"{glob.escape(stem)}[.-]*"):
            leftover.unlink()


def compute_variants_for(encoder, transformer_encoders=TRANSFORMER_ENCODERS):
    return TWO_VARIANTS if encoder in transformer_encoders else ONE_VARIANT


def export_variants(fp32, calibration, variants):
    """({variant: file}, {variant: reason}): a variant that fails (not for lack of disk or memory) is recorded."""
    files, failures = {}, {}
    for variant in variants:
        try:
            files[variant] = to_fp16(fp32) if variant == FP16 else quantize_int8(fp32, calibration, variant=variant)
        except Exception as error:
            if resource_error(error):
                raise
            failures[variant] = f"{type(error).__name__}: {(str(error).splitlines() or [''])[0]}"
    return files, failures


def choose_variant(candidates, threads, bar=FIDELITY_MIN_CORR):
    """(variant, reason): the fastest int8 compute variant whose min_corr reaches bar, else the highest min_corr."""
    if len(candidates) == 1:
        return candidates[0]["variant"], "the only int8 variant measured"
    summary = "; ".join(f"{entry['variant']} {entry['ms']:.2f} ms at {threads} threads, min corr "
                        f"{entry['min_corr']:.4f}" for entry in candidates)
    passing = [entry for entry in candidates if entry["min_corr"] >= bar]
    if passing:
        return min(passing, key=lambda entry: entry["ms"])["variant"], f"fastest with min corr >= {bar}: {summary}"
    return (max(candidates, key=lambda entry: entry["min_corr"])["variant"],
            f"none reaches min corr {bar}, so the highest is taken: {summary}")


def run_matrix(specs, calibration, held_out, onnx_root, out, threads=THREADS, runs=LATENCY_RUNS,
               warmup=WARMUP_RUNS, n_classes=N_CLASSES, embedding_dim=EMBEDDING_DIM, cpu_model=None,
               transformer_encoders=TRANSFORMER_ENCODERS, device=None, retrievers=None):
    """Measures every spec into latency.csv and quantization.csv, then writes the summary files."""
    onnx_root, out = Path(onnx_root), Path(out)
    onnx_root.mkdir(parents=True, exist_ok=True)
    out.mkdir(parents=True, exist_ok=True)
    inputs, quantization, latency = {}, [], []
    for spec in specs:
        key = (spec.size, spec.normalisation)
        if key not in inputs:
            inputs = {cached: arrays for cached, arrays in inputs.items() if cached[0] == spec.size}
            inputs[key] = tuple(calibration_inputs(frame["file"], *key) for frame in (calibration, held_out))
        rows, timings = _measure(spec, inputs[key], onnx_root, threads, runs, warmup, transformer_encoders, device)
        quantization += rows
        latency += timings
        for row in rows:
            built = row["quantization"] != FAILED
            print(f"{spec.name}@{spec.size} {row['variant']}: fp32 {row['fp32_mb']:.2f} MB, "
                  + (f"{row['variant_mb']:.2f} MB {row['quantization']}, min corr {row['min_corr']:.4f} "
                     f"({row['weights']} weights){', chosen int8' if row['chosen'] else ''}" if built
                     else f"FAILED: {row['fallback_reason']}"), flush=True)
    pd.DataFrame(latency).to_csv(out / LATENCY_FILE, index=False)
    pd.DataFrame(quantization).to_csv(out / QUANTIZATION_FILE, index=False)
    write_summary(specs, calibration, held_out, out, threads, runs, warmup, n_classes, embedding_dim, cpu_model,
                  device, retrievers or {})
    return out


def _measure(spec, inputs, onnx_root, threads, runs, warmup, transformer_encoders, device):
    """(quantization rows, latency rows) of one spec."""
    calibration, held_out = inputs
    with model_files(onnx_root, f"{spec.name}_{spec.size}"):
        fp32 = spec.build(onnx_root / f"{spec.name}_{spec.size}.onnx")
        variants = compute_variants_for(spec.encoder, transformer_encoders) + DEPLOYMENT_VARIANTS
        files, failures = export_variants(fp32, calibration, variants)
        latency = _timings(spec, fp32, files, threads, runs, warmup, device)
        quantization = [_quantization_row(spec, fp32, variant, len(held_out), files.get(variant),
                                          fidelity(fp32, files[variant], held_out) if variant in files else None,
                                          failures.get(variant, ""),
                                          detection_scores(run_photos(fp32, held_out), run_photos(files[variant],
                                                                                                held_out))
                                          if spec.role == DETECTOR_ROLE and variant in files else None)
                        for variant in variants]
    top = max(threads)
    medians = {row["variant"]: row["median_ms"] for row in latency
               if row["precision"] != FP32 and row["threads"] == top}
    candidates = [{"variant": row["variant"], "min_corr": row["min_corr"], "ms": medians[row["variant"]]}
                  for row in quantization if row["variant"] in compute_variants_for(spec.encoder, transformer_encoders)
                  and row["quantization"] != FAILED]
    chosen, reason = choose_variant(candidates, top) if candidates else (None, "every int8 compute variant failed")
    return [{**row, "chosen": row["variant"] == chosen, "choice_reason": reason} for row in quantization], latency


def detection_scores(reference, other, conf=YOLO_CONF, iou=DETECTION_IOU):
    """How a YOLO head's top detection survives a variant: detection_agreement = both_empty_share +
    fp32_detection_rate x both_found_agreement, where a found box scores conf or above and boxes match at iou."""
    first, second = (_top_box(next(iter(outputs.values()))) for outputs in (reference, other))
    found, kept = first[1] >= conf, second[1] >= conf
    overlap = _iou(first[0], second[0]) >= iou
    return {"detection_agreement": float(np.mean((found == kept) & (~found | overlap))),
            "fp32_detection_rate": float(found.mean()),
            "both_found_agreement": float(np.mean(kept[found] & overlap[found])) if found.any() else None,
            "both_empty_share": float(np.mean(~found & ~kept))}


def _top_box(head):
    """(centre x, y, width, height of the top box, its score) per photo of a YOLO head (N, 4 + classes, anchors)."""
    scores = head[:, 4:, :].max(axis=1)
    best = scores.argmax(axis=1)
    rows = np.arange(len(head))
    return head[rows, :4, best], scores[rows, best]


def _iou(first, second):
    def corners(boxes):
        return np.stack([boxes[:, 0] - boxes[:, 2] / 2, boxes[:, 1] - boxes[:, 3] / 2,
                         boxes[:, 0] + boxes[:, 2] / 2, boxes[:, 1] + boxes[:, 3] / 2], axis=1)
    a, b = corners(first), corners(second)
    width = np.clip(np.minimum(a[:, 2], b[:, 2]) - np.maximum(a[:, 0], b[:, 0]), 0, None)
    height = np.clip(np.minimum(a[:, 3], b[:, 3]) - np.maximum(a[:, 1], b[:, 1]), 0, None)
    shared = width * height
    union = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]) + (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1]) - shared
    return np.where(union > 0, shared / np.where(union > 0, union, 1), 0.0)


def _quantization_row(spec, fp32, variant, held_out_photos, path=None, check=None, failure="", detections=None):
    """One quantization.csv row; a failed variant has no file, so its file columns are empty."""
    record = quantization_of(path) if path else {"quantization": FAILED, "fallback_reason": failure, "fp32_ops": {},
                                                  ATTENTION_KEY: 0}
    return {"model": spec.name, "role": spec.role, "encoder": spec.encoder, "size": spec.size, "variant": variant,
            "exporter": exporter_of(fp32) or "ultralytics", "opset": opset_of(fp32),
            "variant_opset": opset_of(path) if path else None, "fp32_mb": file_mb(fp32),
            "variant_mb": file_mb(path) if path else None, "quantization": record["quantization"],
            "fallback_reason": record["fallback_reason"], "fp32_ops": json.dumps(record["fp32_ops"]),
            "fp32_nodes": len(fp32_node_names(path)) if path else 0,
            "attention_nodes_kept_fp32": record[ATTENTION_KEY],
            "weights": spec.weights, "min_corr": check["min_corr"] if check else None,
            "top1_agreement": check["top1_agreement"] if check else None,
            **(detections or dict.fromkeys(DETECTION_COLUMNS)),
            "output_corr": json.dumps({name: round(entry["corr"], 6)
                                       for name, entry in check["outputs"].items()}) if check else "{}",
            "held_out_photos": held_out_photos}


def _timings(spec, fp32, variants, threads, runs, warmup, device):
    """One row per file and thread count; median_ms_rerun repeats the measurement in a second fresh session."""
    files = [(FP32, "", "", fp32)] + [(HALF if variant == FP16 else INT8, variant,
                                       quantization_of(path)["quantization"], path)
                                      for variant, path in variants.items()]
    rows = []
    for precision, variant, quantization, path in files:
        for count in threads:
            timing = cpu_latency(path, spec.size, runs, count, warmup)
            rows.append({"model": spec.name, "role": spec.role, "encoder": spec.encoder, "size": spec.size,
                         "device": device, "precision": precision, "variant": variant, "quantization": quantization,
                         "opset": opset_of(path), "threads": count, **timing,
                         "median_ms_rerun": cpu_latency(path, spec.size, runs, count, warmup)["median_ms"],
                         "runs": runs, "warmup": warmup, "load_1min": os.getloadavg()[0]})
    return rows


def prototype_mb(n_classes, dim):
    """int8 class prototypes, one byte per value plus a float32 scale per row, in 10^6 bytes."""
    return (n_classes * dim + n_classes * 4) / MEGABYTE


def package_table(models, prototypes, budget=BUDGET_MB, needs_nms=()):
    """One row per deployable combination against the budget: each SnakeID alone and with each front end at its size
    (a detector also at 640), each F5 SnakeID alone; ms_<n>t columns add up as pipeline_ms_<n>t."""
    models = models.assign(nominal_size=models["nominal_size"] if "nominal_size" in models else models["size"])
    latency = [column for column in models.columns if column.startswith(MS_PREFIX)]
    front_ends = models[models["role"].isin((SEGMENTER, DETECTOR_ROLE))]

    def row(snake, front, label):
        front_mb = 0.0 if front is None else front["int8_mb"]
        total = snake["int8_mb"] + front_mb + prototypes
        nms = [f"pipeline_ms leaves out {front['model']}'s NMS"] if front is not None and front["model"] in needs_nms \
            else []
        return {"encoder": snake["encoder"], "snakeid": snake["model"], "snakeid_int8": snake.get("variant", ""),
                "size": snake["size"], "front_end": label, "snakeid_mb": snake["int8_mb"], "front_end_mb": front_mb,
                "prototypes_mb": prototypes, "total_mb": total, "budget_mb": budget, "headroom_mb": budget - total,
                "fits": total <= budget,
                **{f"pipeline_{column}": snake[column] + (0.0 if front is None else front[column])
                   for column in latency},
                "note": "; ".join([PACKAGE_NOTE] + nms)}

    rows = []
    for snake in models[models["role"] == SNAKEID].to_dict("records"):
        partners = front_ends[(front_ends["size"] == snake["nominal_size"])
                              | ((front_ends["role"] == DETECTOR_ROLE) & (front_ends["size"] == NATIVE_DETECTOR_SIZE))]
        rows.append(row(snake, None, "none"))
        rows += [row(snake, front, f"{front['model']}@{front['size']}") for front in partners.to_dict("records")]
    rows += [row(snake, None, "shared mask head (F5)") for snake in models[models["role"] == SNAKEID_F5].to_dict(
        "records")]
    return pd.DataFrame(rows)


def write_summary(specs, calibration, held_out, out, threads, runs, warmup, n_classes, embedding_dim, cpu_model,
                  device, retrievers):
    """package.csv, calibration_photos.csv and versions.json from the specs' rows in both tables."""
    planned = {(spec.name, spec.size) for spec in specs}
    quantization, latency = (table[[key in planned for key in zip(table["model"], table["size"])]]
                             for table in (pd.read_csv(out / QUANTIZATION_FILE), pd.read_csv(out / LATENCY_FILE)))
    nominal = {(spec.name, spec.size): spec.nominal_size for spec in specs}
    prototypes = prototype_mb(n_classes, embedding_dim)
    blocks = []
    for precision in PACKAGE_PRECISIONS:
        models = _package_models(latency, quantization, nominal, precision)
        block = package_table(models, prototypes, needs_nms=NEEDS_NMS) if len(models) else pd.DataFrame()
        if len(block):
            block.insert(0, "precision", precision)
            blocks.append(block)
    deployment = _deployment_block(latency, quantization, nominal, FP16, prototypes)
    package = pd.concat(blocks + ([deployment] if len(deployment) else []), ignore_index=True) if blocks \
        else pd.DataFrame()
    package = with_fps(package.assign(latency_device=device)) if len(package) else package
    with_retrievers(package, retrievers).to_csv(out / PACKAGE_FILE, index=False)
    used = pd.concat([calibration.assign(role="calibration"), held_out.assign(role="held_out")])
    used[["path", "species", "role"]].to_csv(out / CALIBRATION_FILE, index=False)
    write_json(out / VERSIONS_FILE, {
        **_versions(), "machine": machine_info(cpu_model, device),
        "settings": {"retrievers_mb": retrievers, "sizes": sorted({spec.size for spec in specs}),
                     "threads": list(threads), "runs": runs, "warmup": warmup, "batch": 1,
                     "n_classes": n_classes, "embedding_dim": embedding_dim, "budget_mb": BUDGET_MB,
                     **_sample_record(calibration, held_out),
                     "variants": {"transformer_encoders": list(TRANSFORMER_ENCODERS), "measured": list(TWO_VARIANTS),
                                  "others": list(ONE_VARIANT), "every_model": list(DEPLOYMENT_VARIANTS)},
                     "fps_target": FPS_TARGET, "detection": {"conf": YOLO_CONF, "iou": DETECTION_IOU},
                     "snakeid_precision": FP16,
                     "models_measured": len(set(zip(quantization["model"], quantization["size"]))),
                     "stopped_early": ""}})


def _deployment_block(latency, quantization, nominal, snakeid_precision, prototypes):
    """package_table rows for what would ship: each SnakeID in snakeid_precision with each detector as deployed."""
    snakeids = _package_models(latency, quantization, nominal, snakeid_precision)
    snakeids = snakeids[snakeids["role"] == SNAKEID]
    detectors, chosen = deployed_detectors(latency, quantization, nominal)
    if snakeids.empty:
        return pd.DataFrame()
    block = package_table(pd.concat([snakeids, detectors], ignore_index=True), prototypes, needs_nms=NEEDS_NMS)
    block.insert(0, "precision", DEPLOYMENT)
    scores = {f"{row['model']}@{row['size']}": row for row in quantization.to_dict("records")
              if chosen.get(f"{row['model']}@{row['size']}") == row["variant"]}
    columns = {f"front_end_{column}": [scores[label].get(column) if label in scores else None
                                       for label in block["front_end"]] for column in DETECTION_COLUMNS}
    return block.assign(front_end_precision=block["front_end"].map(chosen).astype(object).fillna(""), **columns)


def deployed_detectors(latency, quantization, nominal):
    """Each detector in its smallest variant with detection_agreement >= DETECTION_BAR, else fp32."""
    rows, chosen = [], {}
    detectors = quantization[(quantization["role"] == DETECTOR_ROLE) & (quantization["quantization"] != FAILED)]
    for (model, size), variants in detectors.groupby(["model", "size"]):
        passing = variants[variants["detection_agreement"] >= DETECTION_BAR].sort_values("variant_mb")
        if len(passing):
            rows.append(_package_models(latency, passing.head(1).assign(chosen=True), nominal, INT8_PACKAGE))
            chosen[f"{model}@{size}"] = passing["variant"].iloc[0]
        else:
            fp32 = variants.head(1)[["model", "role", "encoder", "size", "fp32_mb"]].rename(
                columns={"fp32_mb": "int8_mb"}).merge(_medians(latency[latency["precision"] == FP32],
                                                               ["model", "size"]),
                                                      on=["model", "size"], how="left", validate="one_to_one")
            rows.append(_for_package(fp32, nominal, variant=FP32))
            chosen[f"{model}@{size}"] = FP32
    return (pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()), chosen


def _medians(latency, index):
    """Median ms per index row, one ms_<n>t column per thread count."""
    return latency.pivot_table(index=index, columns="threads", values="median_ms").rename(
        columns=lambda count: f"{MS_PREFIX}{count}t").reset_index()


def _for_package(models, nominal, **columns):
    return models.assign(**columns, encoder=models["encoder"].astype(object).fillna(""),  # all-empty reads as float
                         nominal_size=[nominal[key] for key in zip(models["model"], models["size"])])


def _package_models(latency, quantization, nominal, precision):
    """One package_table row per model and size in one precision (int8: the chosen compute variant)."""
    picked = quantization[quantization["chosen"]] if precision == INT8_PACKAGE else quantization[
        (quantization["variant"] == precision) & (quantization["quantization"] != FAILED)]
    models = picked[["model", "role", "encoder", "size", "variant", "variant_mb"]].rename(
        columns={"variant_mb": "int8_mb"}).merge(_medians(latency[latency["precision"] != FP32],
                                                          ["model", "size", "variant"]),
                                                 on=["model", "size", "variant"], how="left", validate="one_to_one")
    return _for_package(models, nominal)


def with_fps(package, target=FPS_TARGET):
    """Per thread count, the frames per second of pipeline_ms (the models alone) and whether it meets the target."""
    columns = {}
    for column in [name for name in package.columns if name.startswith(f"pipeline_{MS_PREFIX}")]:
        threads = column.removeprefix(f"pipeline_{MS_PREFIX}")
        columns[f"pipeline_fps_{threads}"] = 1000.0 / package[column]
        columns[f"meets_{target}fps_{threads}"] = columns[f"pipeline_fps_{threads}"] >= target
    return package.assign(**columns)


def read_retrievers(path):
    """{retriever: MB} from Q1's retrieval.csv: the embedder, tokenizer and index each retriever puts on the phone."""
    table = pd.read_csv(path)
    rows = table[table["style"] == "all"]
    return dict(zip(rows["retriever"], rows["mb"].astype(float)))


def with_retrievers(package, retrievers, budget=BUDGET_MB):
    """The package table with, per retriever, the total MB with it and whether that fits."""
    if package.empty:
        return package
    columns = {}
    for name, mb in retrievers.items():
        total = package["total_mb"] + mb
        columns[WITH_RETRIEVER_MB.format(name)], columns[WITH_RETRIEVER_FITS.format(name)] = total, total <= budget
    return package.assign(**columns)


def _sample_record(calibration, held_out):
    return {"calibration_photos": len(calibration), "calibration_species": int(calibration["species"].nunique()),
            "calibration_hash": subset_hash(calibration["path"]), "held_out_photos": len(held_out),
            "held_out_species": int(held_out["species"].nunique()),
            "static_int8": {**{name: str(value) for name, value in STATIC_OPTIONS.items()},
                            "calibration_flush": calibration_flush(len(calibration))}}


def _versions():
    from importlib.metadata import version

    return {**library_versions(torch.device("cpu")), "onnxruntime": ort.__version__, "onnx": onnx.__version__,
            "onnxscript": version("onnxscript"), "ultralytics": version("ultralytics"), "exporter": EXPORTER,
            "opset": {"torchscript": OPSET, "torch_export": DYNAMO_OPSET, "per_file": "opset columns"}}


def machine_info(cpu_model=None, device=None):
    """The CPU the latencies were measured on; the CPU model is given, as the laptop's sandbox can't read it."""
    return {"os": platform.platform(), "machine": platform.machine(), "processor": platform.processor() or None,
            "logical_cores": os.cpu_count(), "cpu_model": cpu_model, "latency_device": device}

In [7]:
QUANTISED = "quantised"  # quantized_accuracy's reduced-file side; FP32 is best.pt in torch


def quantized_accuracy(run_dir, onnx_file, frame, classes, train_counts, val_counts=None, num_workers=0):
    """A trained run's final metrics from best.pt in fp32 and from a reduced ONNX copy, with their agreement."""
    cfg, model, _ = load_checkpoint(run_dir)
    session = open_session(onnx_file)
    names = [output.name for output in session.get_outputs()]
    collected = {FP32: defaultdict(list), QUANTISED: defaultdict(list)}
    with torch.inference_mode():
        for batch in eval_loader(cfg, frame, input_mode(cfg), num_workers, torch.device("cpu")):
            reference, reduced = model(batch["image"]), _run_session(session, batch["image"].numpy())
            for output in names:
                collected[FP32][output].append(reference[output].numpy())
                collected[QUANTISED][output].append(reduced[output])
    outputs = {precision: {output: np.concatenate(parts) for output, parts in values.items()}
               for precision, values in collected.items()}
    with np.load(Path(run_dir) / PROTOTYPES_FILE) as saved:
        prototypes = saved["plain"], saved["balanced"]
    taxa = class_tables(classes)
    scores = {precision: final_metrics(frame, values, prototypes, train_counts, taxa, cfg.arcface_s, val_counts)
              for precision, values in outputs.items()}
    comparison = compare_outputs(outputs[FP32], outputs[QUANTISED])
    cosine = np.sum(outputs[FP32][EMBEDDING] * outputs[QUANTISED][EMBEDDING], axis=1)
    labels = frame["class_id"].to_numpy()
    right = outputs[FP32][SPECIES].argmax(axis=1) == labels
    kept = outputs[QUANTISED][SPECIES].argmax(axis=1)[right] == labels[right]
    headline = {precision: {"top1": score["head"]["top1"], "top3": score["head"]["top3"],
                            "macro_f1": score["head"]["macro_f1"], "ece": score["calibration"]["ece"]}
                for precision, score in scores.items()}
    return {"run_id": cfg.run_id, "photos": len(frame), "onnx": str(onnx_file),
            "quantization": quantization_of(onnx_file),
            "val_counts": "the frame's own" if val_counts is None else f"given ({int(np.sum(val_counts))} photos)",
            "headline": headline, **scores,
            "agreement": {"top1_agreement": comparison["top1_agreement"],
                          "kept_correct": float(kept.mean()) if len(kept) else None, "min_corr": comparison["min_corr"],
                          "max_abs_species_logit": comparison["outputs"][SPECIES]["max_abs_diff"],
                          "mean_embedding_cosine": float(cosine.mean())}}


def run_accuracy(run_dir, calibration, held_out, data_root, subset_csv, onnx_root, out, threads=THREADS,
                 runs=LATENCY_RUNS, warmup=WARMUP_RUNS, cpu_model=None, transformer_encoders=TRANSFORMER_ENCODERS,
                 device=None, boxes=None, box_source=None, time_latency=True):
    """accuracy_<run_id>.json: best.pt in each variant scored on the held-out photos, and checked against the run's
    saved outputs. A box-crop run needs `boxes` ({path: box or None})."""
    cfg, model, _ = load_checkpoint(run_dir)
    mode = input_mode(cfg)
    if mode not in (FULL_INPUT, BOX_CROP):
        raise ValueError(f"run {cfg.run_id} reads {mode} input; only {FULL_INPUT} and {BOX_CROP} runs are scored here")
    if mode == BOX_CROP and boxes is None:
        raise ValueError(f"run {cfg.run_id} crops each photo to its front end's box, so it needs boxes")
    onnx_root, out = Path(onnx_root), Path(out)
    onnx_root.mkdir(parents=True, exist_ok=True)
    out.mkdir(parents=True, exist_ok=True)
    frame_cfg = cfg if mode == FULL_INPUT else replace(cfg, front_end="F0", mask_source=SAM,
                                                       box_source="none")  # the boxes are given, not read
    tables = read_tables(frame_cfg, data_root, None, subset_csv)
    frames = run_frames(frame_cfg, tables, data_root)
    train_counts, val_counts = (np.bincount(frame["class_id"], minlength=len(tables.classes))
                                for frame in (frames.train, frames.val))
    val = frames.val[frames.val["path"].isin(held_out["path"])]
    if len(val) != len(held_out):
        raise ValueError(f"{len(held_out) - len(val)} held-out photos aren't in run {cfg.run_id}'s validation set")
    val = val.assign(image_file=val["path"].map(dict(zip(held_out["path"], held_out["file"]))), mask_file="")
    crops = boxes if mode == BOX_CROP else None
    if crops is not None:
        val = val.assign(box=pd.Series([crops.get(path) for path in val["path"]], index=val.index, dtype=object))
    calibration_boxes = None if crops is None else [crops.get(path) for path in calibration["path"]]
    held_out_boxes = None if crops is None else list(val["box"])
    spec = ModelSpec(cfg.run_id, SNAKEID, cfg.encoder, cfg.input_size, IMAGENET, None, cfg.input_size)
    stem = f"{cfg.run_id}_{cfg.input_size}"
    with model_files(onnx_root, stem):
        fp32 = to_onnx(model, onnx_root / f"{stem}.onnx", cfg.input_size)
        arrays = calibration_inputs(calibration["file"], cfg.input_size, IMAGENET, calibration_boxes)
        files, failures = export_variants(fp32, arrays, compute_variants_for(cfg.encoder, transformer_encoders)
                                          + DEPLOYMENT_VARIANTS)
        variants = {variant: quantized_accuracy(run_dir, path, val, tables.classes, train_counts, val_counts)
                    for variant, path in files.items()}
        with np.load(Path(run_dir) / VAL_OUTPUTS) as saved:
            saved_logits = dict(zip(saved["paths"], saved[SPECIES]))
        local = run_photos(fp32, calibration_inputs(val["image_file"], cfg.input_size, IMAGENET, held_out_boxes))
        report = {
            "run_id": cfg.run_id, "photos": len(val),
            "val_counts": {"source": f"run {cfg.run_id}'s full validation set", "photos": int(val_counts.sum())},
            "variants": variants, "failed": failures,
            "saved_outputs_top1_agreement": float(np.mean([saved_logits[path].argmax() == top1 for path, top1
                                                           in zip(val["path"], local[SPECIES].argmax(axis=1))])),
            "saved_outputs_max_abs_logit": float(max(np.abs(saved_logits[path] - logits).max()
                                                     for path, logits in zip(val["path"], local[SPECIES]))),
            "boxes": {"source": box_source or (FULL_INPUT if crops is None else "given"),
                      "held_out_with_box": _box_share(held_out_boxes),
                      "calibration_with_box": _box_share(calibration_boxes)},
            "latency": _timings(spec, fp32, files, threads, runs, warmup, device) if time_latency else [],
            "exporter": exporter_of(fp32), "opset": opset_of(fp32), "fp32_mb": file_mb(fp32),
            "variant_mb": {variant: file_mb(path) for variant, path in files.items()}, "versions": _versions(),
            "machine": machine_info(cpu_model, device), "sample": _sample_record(calibration, held_out)}
    return write_json(out / ACCURACY_FILE.format(cfg.run_id), report)


def _box_share(boxes):
    return 0.0 if boxes is None else float(np.mean([not is_missing_box(box) for box in boxes]))


def detector_boxes(weights, photos, calibration, onnx_root, input_size=DETECTOR_SIZE):
    """{path: box or None} from the app's box finder running the fine-tuned detector's int8_weights ONNX file."""
    from ultralytics import YOLO

    onnx_root = Path(onnx_root)
    onnx_root.mkdir(parents=True, exist_ok=True)
    found = {}
    with model_files(onnx_root, "detector"):
        fp32 = ultralytics_onnx(weights, YOLO_CONFIGS[FINE_TUNED][1], onnx_root / "detector.onnx", input_size)
        detector = YOLO(str(quantize_int8(fp32, calibration_inputs(calibration["file"], input_size, UNIT),
                                          variant=INT8_WEIGHTS)), task="detect")
        for start in range(0, len(photos), DETECTOR_CHUNK):
            chunk = photos.iloc[start:start + DETECTOR_CHUNK]
            hits = predict_boxes(detector, [load_rgb(file) for file in chunk["file"]])
            found.update({path: None if hit is None else tuple(float(value) for value in hit[0])
                          for path, hit in zip(chunk["path"], hits)})
    return found

In [8]:
OWNER = "puneetbajajiitm"
ON_KAGGLE = Path("/kaggle/working").exists()
RETRIEVERS = CODE.parent / "results" / "m3" / "q1" / "retrieval.csv"  # Q1's retriever sizes, for the package totals


def input_root(slug):
    """Where an attached dataset is mounted; Kaggle no longer mounts datasets at /kaggle/input/<slug>."""
    import kagglehub

    return Path(kagglehub.dataset_download(f"{OWNER}/{slug}"))


DATASET_PREFIX = "dataset:"  # names a dataset, e.g. a private copy of another account's kernel output


def kernel_output(handle):
    """A local folder as it is, a kernel's output folder, or for "dataset:owner/slug" that dataset's files."""
    if Path(handle).is_dir():
        return Path(handle)
    import kagglehub

    if handle.startswith(DATASET_PREFIX):
        return Path(kagglehub.dataset_download(handle.removeprefix(DATASET_PREFIX)))
    return Path(kagglehub.notebook_output_download(handle))


def run_folder(handle, run_id):
    """The one folder named run_id holding best.pt, anywhere under the kernel output or dataset."""
    found = sorted(path.parent for path in kernel_output(handle).rglob(CHECKPOINT_FILE) if path.parent.name == run_id)
    assert len(found) == 1, f"{handle} must hold one {run_id} folder with {CHECKPOINT_FILE}; found {found}"
    return found[0]


def cpu_model():
    """The CPU's model name, as /proc/cpuinfo gives it."""
    cpuinfo = Path("/proc/cpuinfo")
    lines = cpuinfo.read_text().splitlines() if cpuinfo.exists() else []
    names = [line.split(":", 1)[1].strip() for line in lines if line.startswith("model name")]
    return names[0] if names else platform.processor() or None


CPU_MODEL = 'Apple M4 Pro'
DEVICE = "Kaggle CPU" if ON_KAGGLE else f"laptop CPU ({CPU_MODEL})"  # the laptop is the O6 device; Kaggle the reference
assert ON_KAGGLE or OUT_DIR, "off Kaggle, set OUT_DIR"
OUT = Path("/kaggle/working/x1") if ON_KAGGLE else Path(OUT_DIR)
# scratch for the ONNX files, outside the saved output; each model's files go once it is measured
ONNX_ROOT = Path("/kaggle/tmp/x1_onnx") if ON_KAGGLE else OUT.with_name(f"{OUT.name}_onnx")
DATA = Path(DATA_DIR) if DATA_DIR else input_root("serpentguard-indian-snakes-clean")
SUBSET = Path(SUBSET_CSV) if SUBSET_CSV else input_root("serpentguard-m3-subset") / "subset.csv"
if SAMPLE_DIR:
    SAMPLE = Path(SAMPLE_DIR)
else:
    manifests = sorted(kernel_output(SAMPLE_KERNEL).rglob(MANIFEST))
    assert len(manifests) == 1, f"{SAMPLE_KERNEL}'s output must hold one {MANIFEST}; found {manifests}"
    SAMPLE = manifests[0].parent
calibration, held_out = read_sample(SAMPLE)  # 200 training photos under train/, validation photos under val/
absent = [file for file in [*calibration["file"], *held_out["file"]] if not Path(file).is_file()]
assert not absent, f"{len(absent)} sample photos are missing, e.g. {absent[0]}"

runs = {run_id: run_folder(handle, run_id) for handle, run_id in ACCURACY_RUNS}

for required in (DATA / "metadata.csv", DATA / "classes.csv", SUBSET, RETRIEVERS):
    assert required.exists(), f"missing input {required}"
retrievers = read_retrievers(RETRIEVERS)
specs = [spec for spec in model_specs(SIZES, trained={FINE_TUNED: DETECTOR} if DETECTOR else None)
         if not SPECS or spec.name in SPECS]
unknown = sorted(set(SPECS or ()) - {spec.name for spec in specs})
assert not unknown, f"SPECS names models outside the matrix: {unknown}"
for folder in (OUT, ONNX_ROOT):
    folder.mkdir(parents=True, exist_ok=True)
print(f"CPU: {CPU_MODEL}, {os.cpu_count()} logical cores; latency labelled {DEVICE!r}")
print(f"sample: {SAMPLE}, {len(calibration)} calibration and {len(held_out)} validation photos")
print(f"{len(specs)} model x size specs; accuracy runs:",
      ", ".join(f"{name} ({folder})" for name, folder in runs.items()))
print("retrievers (MB):", {name: round(mb, 2) for name, mb in retrievers.items()})

CPU: Apple M4 Pro, 14 logical cores; latency labelled 'laptop CPU (Apple M4 Pro)'
sample: ../../kaggle/m3_runs/local_sample/out2/local_sample, 200 calibration and 300 validation photos
5 model x size specs; accuracy runs: H1_top1_s1 (../../kaggle/m3_runs/stageC/final_candidates/top1_s1/H1_top1_s1)
retrievers (MB): {'R0': 0.04, 'R1': 0.21, 'R2-L6': 22.94, 'R2-L3': 17.69}


In [9]:
started = time.time()
run_matrix(specs, calibration, held_out, ONNX_ROOT, OUT, THREADS, LATENCY_RUNS, cpu_model=CPU_MODEL, device=DEVICE,
           retrievers=retrievers)
print(f"matrix: {(time.time() - started) / 60:.1f} min")

snakeid_mobilenetv4_conv_small@320 static_qdq: fp32 6.26 MB, 1.87 MB static_qdq, min corr 0.9792 (untrained weights), chosen int8


snakeid_mobilenetv4_conv_small@320 int8_weights: fp32 6.26 MB, 1.68 MB int8_weights, min corr 0.9998 (untrained weights)


snakeid_mobilenetv4_conv_small@320 fp16: fp32 6.26 MB, 3.15 MB fp16, min corr 1.0000 (untrained weights)


snakeid_efficientnet_b0@320 static_qdq: fp32 17.51 MB, 5.28 MB static_qdq, min corr 0.9862 (untrained weights), chosen int8


snakeid_efficientnet_b0@320 int8_weights: fp32 17.51 MB, 4.66 MB int8_weights, min corr 0.9994 (untrained weights)


snakeid_efficientnet_b0@320 fp16: fp32 17.51 MB, 8.82 MB fp16, min corr 1.0000 (untrained weights)


... (7 outputs left out)

yolo26n-ft@640 static_qdq: fp32 9.81 MB, 3.07 MB static_qdq, min corr 0.9998 (trained: detector.pt weights), chosen int8


yolo26n-ft@640 int8_weights: fp32 9.81 MB, 2.77 MB int8_weights, min corr 1.0000 (trained: detector.pt weights)


yolo26n-ft@640 fp16: fp32 9.81 MB, 4.97 MB fp16, min corr 1.0000 (trained: detector.pt weights)


matrix: 7.5 min


In [10]:
from serpentguard.frontend import BOX_FLOOR

boxes = source = None
if DETECTOR:  # box-crop runs are scored on the deployed detector's crops: its int8_weights file, as the app runs it
    boxes = detector_boxes(DETECTOR, pd.concat([calibration, held_out]), calibration, ONNX_ROOT, DETECTOR_SIZE)
    source = (f"detector {Path(DETECTOR).name} as {INT8_WEIGHTS} at {DETECTOR_SIZE}, its top box at confidence >= "
              f"{BOX_FLOOR} (frontend.predict_boxes)")
for run_id, folder in runs.items():
    box_crop = input_mode(load_checkpoint(folder)[0]) == BOX_CROP  # without DETECTOR, run_accuracy stops it
    report = run_accuracy(folder, calibration, held_out, DATA, SUBSET, ONNX_ROOT, OUT, THREADS, LATENCY_RUNS,
                          cpu_model=CPU_MODEL, device=DEVICE, boxes=boxes if box_crop else None,
                          box_source=source if box_crop else None)
    print(json.dumps({"run_id": report["run_id"], "photos": report["photos"],
                      "saved_outputs_top1_agreement": report["saved_outputs_top1_agreement"],
                      **{variant: {"quantization": entry["quantization"]["quantization"],
                                   "headline": entry["headline"], "agreement": entry["agreement"]}
                         for variant, entry in report["variants"].items()}}, indent=2))

{
  "run_id": "H1_top1_s1",
  "photos": 300,
  "saved_outputs_top1_agreement": 0.92,
  "static_qdq_attention_fp32": {
    "quantization": "static_qdq_attention_fp32",
    "headline": {
      "fp32": {
        "top1": 0.36333333333333334,
        "top3": 0.5466666666666666,
        "macro_f1": 0.3156926406926407,
        "ece": 0.4071691385401025
      },
      "quantised": {
        "top1": 0.05,
        "top3": 0.12,
        "macro_f1": 0.028308043085148202,
        "ece": 0.5144745283251232
      }
    },
    "agreement": {
      "top1_agreement": 0.09333333333333334,
      "kept_correct": 0.09174311926605505,
      "min_corr": 0.2136731347390277,
      "max_abs_species_logit": 27.408252716064453,
      "mean_embedding_cosine": 0.21327084302902222
    }
  },
  "dynamic": {
    "quantization": "dynamic",
    "headline": {
      "fp32": {
        "top1": 0.36333333333333334,
        "top3": 0.5466666666666666,
        "macro_f1": 0.3156926406926407,
        "ece": 0.4071691385401025
  

In [11]:
versions = json.loads((OUT / VERSIONS_FILE).read_text())
assert versions["machine"]["latency_device"] == DEVICE and not versions["settings"]["stopped_early"]
assert versions["settings"]["models_measured"] == len(specs), versions["settings"]
latency = pd.read_csv(OUT / LATENCY_FILE)
assert set(latency["device"]) == {DEVICE}

quantization = pd.read_csv(OUT / QUANTIZATION_FILE)
print(quantization[["model", "size", "variant", "quantization", "fp32_mb", "variant_mb", "weights", "min_corr",
                    "top1_agreement", "detection_agreement", "chosen", "fallback_reason"]].to_string())

package = pd.read_csv(OUT / PACKAGE_FILE)
with_retriever = [column for name in retrievers
                  for column in (WITH_RETRIEVER_MB.format(name), WITH_RETRIEVER_FITS.format(name))]
print(f"\npackage against {versions['settings']['budget_mb']} MB; pipeline_ms timed on {DEVICE}")
print(package[["precision", "snakeid", "snakeid_int8", "size", "front_end", "total_mb", "fits", *with_retriever,
               *[column for column in package.columns if column.startswith("pipeline_ms")]]].to_string())

rows = []
for run_id in runs:
    report = json.loads((OUT / ACCURACY_FILE.format(run_id)).read_text())
    for variant, scored in report["variants"].items():
        for side in ("fp32", "quantised"):
            rows.append({"run": run_id, "variant": variant, "side": side, **scored["headline"][side],
                         "top-1 agreement": scored["agreement"]["top1_agreement"],
                         "kept correct": scored["agreement"]["kept_correct"],
                         "embedding cosine": scored["agreement"]["mean_embedding_cosine"],
                         "MB": report["variant_mb"][variant], "photos": scored["photos"]})
    for variant, reason in report["failed"].items():
        print(f"{run_id} {variant} failed: {reason}")
pd.DataFrame(rows).round(4)

                             model  size                    variant               quantization    fp32_mb  variant_mb               weights  min_corr  top1_agreement  detection_agreement  chosen  fallback_reason
0   snakeid_mobilenetv4_conv_small   320                 static_qdq                 static_qdq   6.258021    1.872878             untrained  0.979153        0.770000                  NaN    True              NaN
1   snakeid_mobilenetv4_conv_small   320               int8_weights               int8_weights   6.258021    1.677756             untrained  0.999772        0.990000                  NaN   False              NaN
2   snakeid_mobilenetv4_conv_small   320                       fp16                       fp16   6.258021    3.153874             untrained  0.999999        0.996667                  NaN   False              NaN
3          snakeid_efficientnet_b0   320                 static_qdq                 static_qdq  17.506904    5.278239             untrained  0.986249   

,run,variant,side,top1,top3,macro_f1,ece,top-1 agreement,kept correct,embedding cosine,MB,photos
0,H1_top1_s1,static_qdq_attention_fp32,fp32,0.3633,0.5467,0.3157,0.4072,0.0933,0.0917,0.2133,6.0829,300
1,H1_top1_s1,static_qdq_attention_fp32,quantised,0.0500,0.1200,0.0283,0.5145,0.0933,0.0917,0.2133,6.0829,300
2,H1_top1_s1,dynamic,fp32,0.3633,0.5467,0.3157,0.4072,0.0800,0.0826,0.2042,5.6318,300
3,H1_top1_s1,dynamic,quantised,0.0367,0.0767,0.0181,0.5053,0.0800,0.0826,0.2042,5.6318,300
4,H1_top1_s1,int8_weights,fp32,0.3633,0.5467,0.3157,0.4072,0.9233,0.9633,0.9933,5.6328,300
5,H1_top1_s1,int8_weights,quantised,0.3633,0.5500,0.3144,0.4043,0.9233,0.9633,0.9933,5.6328,300
6,H1_top1_s1,fp16,fp32,0.3633,0.5467,0.3157,0.4072,1.0000,1.0000,1.0000,10.6671,300
7,H1_top1_s1,fp16,quantised,0.3633,0.5467,0.3157,0.4071,1.0000,1.0000,1.0000,10.6671,300
